# Agente inteligente de monitoreo geriátrico remoto

Este cuaderno implementa un agente inteligente diseñado para el monitoreo clínico de pacientes adultos mayores con hipertensión y diabetes. Utilizando una arquitectura basada en LangChain (ReAct) y Telegram, el sistema actúa como un asistente clínico que procesa lenguaje natural para realizar registros, consultas y generar informes médicos detallados.

Características clave:

* Gestión inteligente de pacientes: El agente permite listar, agregar, seleccionar y eliminar pacientes de manera dinámica a través de instrucciones naturales.
* Procesamiento clínico: Incluye 9 herramientas especializadas (@tool) para el registro de signos vitales (presión arterial y glucosa), medicación, síntomas y perfiles clínicos.
* Evaluación de riesgo: Realiza un análisis clínico automatizado del estado de salud global y detecta patrones de riesgo.
* Reportes automatizados: Genera informes clínicos completos acompañados de gráficos de evolución, facilitando la teleconsulta.

# Arquitectura

```
Usuario (Telegram)
    │
    ▼
MessageHandler
    │
    ▼
AgentExecutor (LangChain ReAct)
    │  ├── ConversationBufferMemory  (Historial por chat)
    │  └── 8 herramientas @tool      (Acciones clínicas + gestión de pacientes)
    │
    ▼
Respuesta en lenguaje natural → Telegram

Persistencia:
    Usuario/usuario_<user_id>.pkl  ← un archivo por usuario de Telegram
    (mismo patrón que CasoC_bot)
```


## Librerias y dependencias

In [ ]:
!pip -q install "requests==2.32.4" -q
!pip -q install python-telegram-bot nest_asyncio matplotlib numpy pytz -q
!pip -q install "langchain==0.2.16" "langchain-core==0.2.38" "langchain-openai==0.1.23" "langchain-community==0.2.16" langchain-google-genai -q

In [ ]:
# Librerías estándar
import os
import re as _re
import pickle
import logging
import nest_asyncio
import matplotlib.pyplot as plt
from datetime import datetime
import pytz
import asyncio

# Telegram
from telegram import Update, InputFile, Bot
from telegram.ext import (
    ContextTypes,
    ApplicationBuilder,
    CommandHandler,
    MessageHandler,
    filters,
)

# LangChain
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain.agents import AgentExecutor, create_react_agent
from langchain.memory import ConversationBufferMemory, ConversationBufferWindowMemory
from langchain_openai import ChatOpenAI

# Configuración del logger
nest_asyncio.apply()
logging.basicConfig(
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)

# Otros
from IPython.display import display, Image as IPImage
from contextvars import ContextVar

## Credenciales

In [ ]:
# Token de Telegram
try:
    TOKEN = userdata.get("TELEGRAM_TOKEN")
    if not TOKEN:
        raise ValueError("Secreto vacío.")
    logger.info("✅ Token Telegram cargado correctamente.")
except Exception as e:
    TOKEN = ""
    logger.error(f"❌ TELEGRAM_TOKEN no encontrado: {e}")

# API key de OpenRouter
try:
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    if not OPENROUTER_API_KEY:
        raise ValueError("Secreto 'OPENROUTER_API_KEY' vacío o sin acceso activado.")
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
    logger.info("✅ API Key de OpenRouter cargada correctamente.")
except Exception as e:
    OPENROUTER_API_KEY = ""
    os.environ["OPENROUTER_API_KEY"] = ""
    logger.error(
        f"❌ No se pudo cargar 'OPENROUTER_API_KEY': {e}\n"
    )

## Constantes clínicas y estado global de sesión

In [ ]:
# Constantes clínicas (OMS / ADA)
UMBRAL_SISTOLICA_MAX  = 140
UMBRAL_SISTOLICA_MIN  = 110
UMBRAL_DIASTOLICA_MAX = 90
UMBRAL_DIASTOLICA_MIN = 60
UMBRAL_GLUCOSA_MAX    = 180
UMBRAL_GLUCOSA_MIN    = 70
ALARMAS_NECESARIAS    = 3

SINTOMAS_COMUNES = [
    "Mareos", "Fatiga", "Visión borrosa",
    "Dolor de cabeza", "Náuseas", "Palpitaciones",
]

MEDICAMENTOS_DICT = {
    "losartan"     : {"categoria": "hipertension"},
    "enalapril"    : {"categoria": "hipertension"},
    "amlodipino"   : {"categoria": "hipertension"},
    "metformina"   : {"categoria": "diabetes"},
    "insulina"     : {"categoria": "diabetes"},
    "glibenclamida": {"categoria": "diabetes"},
}

# Persistencia en almacenamiento local
# Ruta base donde se guardan todos los archivos de datos.
BASE_DIR = os.environ.get("DATOS_DIR", os.path.join(os.getcwd(), "datos"))
CARPETA_USUARIOS = os.path.join(BASE_DIR, "Usuarios") + os.sep

def _ruta_pkl(user_id: int) -> str:
    """Retorna la ruta completa del archivo .pkl de un usuario (igual que CasoC_bot)."""
    return f'{CARPETA_USUARIOS}usuario_{user_id}.pkl'

# Estado global de sesión por usuario
# Sesiones guarda la sesión de CADA usuario identificado por su user_id.
# Esto permite que múltiples usuarios usen el bot simultáneamente sin
# interferirse entre sí.
SESIONES: dict = {}   # {user_id: {"pacientes": {}, "paciente_activo": None}}

# Memoria conversacional por chat
MEMORIAS = {}   # {chat_id: ConversationBufferMemory}
AGENTES  = {}   # {chat_id: AgentExecutor}

# ContextVar para aislar chat_id en tareas asyncio concurrentes
_CHAT_ID_CTX: ContextVar[int] = ContextVar("_CHAT_ID_CTX", default=-1)
# FIX: default=-1 como centinela para detectar contextos sin user_id seteado
_USER_ID_CTX: ContextVar[int] = ContextVar("_USER_ID_CTX", default=-1)

print("Constantes y estado global listos")

## Helpers

In [ ]:
def ahora() -> str:
    """Retorna la hora actual en zona Lima, formato día/mes HH:MM."""
    return datetime.now(pytz.timezone("America/Lima")).strftime("%d/%m %H:%M")

# Persistencia en almacenamiento local

def guardar_sesion(user_id: int) -> None:
    """
    Serializa la sesión del usuario en su propio archivo pkl en almacenamiento local.
    Misma lógica que guardar_datos_en_local() del CasoC_bot:
      - Crea la carpeta Usuarios/ si no existe.
      - Guarda solo los datos de ESE usuario.
    """
    try:
        if not os.path.exists(CARPETA_USUARIOS):
            os.makedirs(CARPETA_USUARIOS)
            print(f"Carpeta creada: {CARPETA_USUARIOS}")
        ruta = _ruta_pkl(user_id)
        ruta_tmp = ruta + ".tmp"
        # FIX: escritura atómica - evita corrupción del .pkl si el proceso se interrumpe
        with open(ruta_tmp, 'wb') as f:
            pickle.dump(SESIONES.get(user_id, {"pacientes": {}, "paciente_activo": None}), f)
            f.flush()
            os.fsync(f.fileno())
        # Backup del archivo anterior
        if os.path.exists(ruta):
            import shutil
            shutil.copy2(ruta, ruta + ".bak")
        os.replace(ruta_tmp, ruta)  # atómico en sistemas POSIX
        logger.info(f"Sesión guardada en local → {ruta}")
    except Exception as e:
        logger.error(f"No se pudo guardar en local (user {user_id}): {e}")


def cargar_sesion(user_id: int) -> bool:
    """
    Carga el archivo pkl del usuario desde almacenamiento local hacia SESIONES[user_id].
    Igual que cargar_datos_desde_local() del CasoC_bot.
    Retorna True si había datos previos, False si es usuario nuevo.
    """
    ruta = _ruta_pkl(user_id)
    if not os.path.exists(ruta):
        SESIONES[user_id] = {"pacientes": {}, "paciente_activo": None}
        return False
    try:
        with open(ruta, 'rb') as f:
            datos = pickle.load(f)
        if isinstance(datos, dict) and 'pacientes' in datos:
            SESIONES[user_id] = datos
            logger.info(f"Sesión cargada desde local ← {ruta}")
            return True
    except Exception as e:
        logger.error(f"Error al cargar pkl de user {user_id}: {e}")
    SESIONES[user_id] = {"pacientes": {}, "paciente_activo": None}
    return False


def _sesion(user_id: int) -> dict:
    """Retorna (o inicializa) la sesión del usuario dado."""
    if user_id not in SESIONES:
        SESIONES[user_id] = {"pacientes": {}, "paciente_activo": None}
    return SESIONES[user_id]


# Helpers clínicos

def _datos_paciente(user_id: int | None = None) -> dict:
    """Devuelve el dict de datos del paciente activo del usuario.
    Si user_id es None, lee desde el ContextVar _USER_ID_CTX."""
    uid  = user_id if user_id is not None else _USER_ID_CTX.get()
    # FIX: detectar contexto sin user_id (valor centinela -1)
    if uid == -1:
        raise ValueError("ContextVar _USER_ID_CTX no inicializado. Posible llamada fuera de contexto asyncio.")
    ses  = _sesion(uid)
    nombre = ses.get("paciente_activo")
    if not nombre or nombre not in ses["pacientes"]:
        raise ValueError("No hay paciente activo. Selecciona o crea uno primero con /start.")
    return ses["pacientes"][nombre]


def _nombre_pac(user_id: int | None = None) -> str:
    """Retorna el nombre del paciente activo del usuario o '-'."""
    uid = user_id if user_id is not None else _USER_ID_CTX.get()
    return _sesion(uid).get("paciente_activo") or "-"


def _registrar_alerta(tipo: str, mensaje: str, critica: bool = False, user_id: int | None = None):
    """Guarda una alerta clínica en el paciente activo del usuario."""
    uid = user_id if user_id is not None else _USER_ID_CTX.get()
    try:
        _datos_paciente(uid)["alertas"].append({
            "fecha"  : ahora(),
            "tipo"   : tipo,
            "mensaje": mensaje,
            "critica": critica,
        })
        logger.warning(f"[ALERTA] {tipo} - {mensaje}")
    except Exception:
        pass


def crear_paciente(nombre: str, user_id: int | None = None) -> tuple[bool, str]:
    """Crea un paciente nuevo en la sesión del usuario. Retorna (es_nuevo, nombre_normalizado)."""
    uid    = user_id if user_id is not None else _USER_ID_CTX.get()
    ses    = _sesion(uid)
    nombre = nombre.strip().title()
    if nombre not in ses["pacientes"]:
        ses["pacientes"][nombre] = {
            "perfil"      : {},
            "presion"     : [],
            "glucosa"     : [],
            "medicamentos": [],
            "sintomas"    : [],
            "alertas"     : [],
        }
        return True, nombre
    return False, nombre


def eliminar_paciente(nombre: str, user_id: int | None = None) -> bool:
    """Elimina un paciente de la sesión del usuario. Retorna True si existía."""
    uid    = user_id if user_id is not None else _USER_ID_CTX.get()
    ses    = _sesion(uid)
    nombre = nombre.strip().title()
    if nombre in ses["pacientes"]:
        del ses["pacientes"][nombre]
        if ses.get("paciente_activo") == nombre:
            # Si era el activo, resetear
            restantes = list(ses["pacientes"].keys())
            ses["paciente_activo"] = restantes[0] if restantes else None
        return True
    return False


def listar_pacientes(user_id: int | None = None) -> list[str]:
    """Retorna la lista de nombres de pacientes del usuario."""
    uid = user_id if user_id is not None else _USER_ID_CTX.get()
    return list(_sesion(uid).get("pacientes", {}).keys())


def seleccionar_paciente(nombre: str, user_id: int | None = None) -> tuple[bool, str]:
    """Cambia el paciente activo. Retorna (éxito, nombre_normalizado)."""
    uid    = user_id if user_id is not None else _USER_ID_CTX.get()
    ses    = _sesion(uid)
    nombre = nombre.strip().title()
    if nombre in ses["pacientes"]:
        ses["paciente_activo"] = nombre
        return True, nombre
    return False, nombre


def get_memoria(chat_id: int) -> ConversationBufferMemory:
    """Retorna la memoria conversacional del chat. La crea si no existe.
    FIX: usa ventana deslizante de 10 turnos para evitar desbordamiento del contexto del LLM.
    """
    if chat_id not in MEMORIAS:
        MEMORIAS[chat_id] = ConversationBufferWindowMemory(
            memory_key="chat_history",
            return_messages=False,
            k=10,
        )
    return MEMORIAS[chat_id]


def _get_chat_id_actual() -> int:
    """
    FIX v9: recupera el chat_id real desde dos fuentes, en orden de confiabilidad.

    Por qué es necesario:
      LangChain ejecuta las herramientas @tool en un ThreadPoolExecutor.
      Los ContextVar de Python NO se propagan automáticamente a esos threads,
      así que _CHAT_ID_CTX.get() devuelve -1 dentro de cualquier herramienta.

    Solución:
      1. Intentar leer _CHAT_ID_CTX (funciona en el handler asyncio y en el bucle local).
      2. Si devuelve -1, leer _sesion(uid)["_chat_id"], que manejar_mensaje
         guarda justo antes de llamar agente.invoke() y sí es accesible
         desde cualquier thread porque es un dict Python en memoria.
    """
    chat_id = _CHAT_ID_CTX.get()
    if chat_id != -1:
        return chat_id
    # Fallback: leer desde la sesión del usuario actual
    uid = _USER_ID_CTX.get()
    if uid != -1:
        return _sesion(uid).get("_chat_id", -1)
    return -1

print("Helpers listos")

## Herramientas `@tool` - Lógica clínica

Las 9 herramientas encapsulan la lógica de negocio.
Las `docstrings` son lo que LangChain lee para decidir cuándo y cómo invocar cada herramienta.

| # | Herramienta | Función |
|---|-------------|---------|
| 1 | `registrar_presion` | Guarda presión arterial con alertas clínicas |
| 2 | `registrar_glucosa` | Guarda glucosa en sangre con clasificación |
| 3 | `registrar_medicamento` | Registra toma de medicamento |
| 4 | `registrar_sintoma` | Registra síntoma con intensidad |
| 5 | `consultar_historial` | Resume presión, glucosa, medicamentos, síntomas |
| 6 | `evaluar_riesgo_clinico` | Evalúa nivel de riesgo global |
| 7 | `generar_informe_medico` | Informe clínico completo + gráfico |
| 8 | `gestionar_pacientes` | Listar, agregar, cambiar, eliminar pacientes |
| 9 | `registrar_perfil` | Registrar perfil del paciente |

### 1 Registrar presión arterial

In [ ]:
@tool
def registrar_presion(valores: str) -> str:
    """
    Registra la presión arterial del paciente activo.
    Úsala cuando el usuario mencione "presión", "presión arterial", "tensión"
    o indique dos valores numéricos típicos de este signo vital (ej. '120/80', '130 sobre 85').
    IMPORTANTE: Activa esta herramienta incluso si el usuario solo dice "presión" y omite la palabra "arterial".
    El parámetro debe contener los dos números. Idealmente separados por '/'.
    Ejemplos de input válido: '145/90', '120/80' o '120 sobre 80'.
    """
    try:
        uid   = _USER_ID_CTX.get()
        datos = _datos_paciente(uid)
        nums  = _re.findall(r'\d+', valores)
        if len(nums) < 2:
            return (
                "❌ No pude extraer dos valores numéricos de la entrada recibida.\n"
                f"   Recibí: '{valores}'\n"
                "   Usa el formato 'sistólica/diastólica'. Ejemplo: '145/90'."
            )
        s, d = int(nums[0]), int(nums[1])
        if not (40 <= s <= 300):
            return f"❌ Sistólica {s} mmHg fuera de rango clínico (40–300)."
        if not (30 <= d <= 160):
            return f"❌ Diastólica {d} mmHg fuera de rango clínico (30–160)."

        fecha = ahora()
        datos["presion"].append({"fecha": fecha, "sistolica": s, "diastolica": d})
        total = len(datos["presion"])

        alerta_txt = ""
        if s >= UMBRAL_SISTOLICA_MAX or d >= UMBRAL_DIASTOLICA_MAX:
            alerta_txt = " ⚠️ Valor elevado - posible hipertensión."
            critica = sum(
                1 for r in datos["presion"]
                if r["sistolica"] >= UMBRAL_SISTOLICA_MAX or r["diastolica"] >= UMBRAL_DIASTOLICA_MAX
            ) >= ALARMAS_NECESARIAS
            _registrar_alerta("presion_alta", f"Presión {s}/{d} mmHg en {fecha}", critica, uid)
        elif s < UMBRAL_SISTOLICA_MIN or d < UMBRAL_DIASTOLICA_MIN:
            alerta_txt = " ⚠️ Valor bajo - posible hipotensión."
            _registrar_alerta("presion_baja", f"Presión {s}/{d} mmHg en {fecha}", False, uid)

        logger.info(f"Presión {s}/{d} mmHg - {_nombre_pac(uid)}")
        guardar_sesion(uid)
        return (
            f"✅ Presión registrada [{fecha}]\n"
            f"{s}/{d} mmHg | Total registros: {total}{alerta_txt}"
        )
    except ValueError as e:
        return f"❌ {e}"
    except Exception as e:
        return f"❌ Error inesperado al registrar presión: {e}"

###  2 Registrar glucosa

In [ ]:
@tool
def registrar_glucosa(valor: str) -> str:
    """
    Registra el nivel de glucosa en sangre del paciente activo en mg/dL.
    Úsala cuando el usuario mencione azúcar en sangre, glucemia, glucosa,
    o indique un valor numérico de glucosa como '95', '210', '75 mg/dL'.
    El parámetro debe ser un número (puede incluir decimales).
    Ejemplo de input válido: '95' o '210.5'.
    """
    try:
        uid   = _USER_ID_CTX.get()
        datos = _datos_paciente(uid)
        v = float(valor.replace(",", ".").replace(" ", "").replace("mg/dL", "").strip())
        if not (20 <= v <= 600):
            return f"❌ Valor {v} mg/dL fuera de rango clínico (20–600)."

        fecha = ahora()
        datos["glucosa"].append({"fecha": fecha, "valor": v})
        total = len(datos["glucosa"])

        if v < UMBRAL_GLUCOSA_MIN:
            estado_cl = "⚠️ Hipoglucemia"
            critica = sum(1 for r in datos["glucosa"] if r["valor"] < UMBRAL_GLUCOSA_MIN) >= 2
            _registrar_alerta("glucosa_baja", f"Glucosa {v} mg/dL en {fecha}", critica, uid)
        elif v < 100:
            estado_cl = "✅ Normal en ayunas"
        elif v < 140:
            estado_cl = "✅ Normal postprandial"
        elif v < UMBRAL_GLUCOSA_MAX:
            estado_cl = "🟡 Elevada - monitorear"
        else:
            estado_cl = "🔴 Hiperglucemia"
            critica = sum(1 for r in datos["glucosa"] if r["valor"] >= UMBRAL_GLUCOSA_MAX) >= ALARMAS_NECESARIAS
            _registrar_alerta("glucosa_alta", f"Glucosa {v} mg/dL en {fecha}", critica, uid)

        logger.info(f"Glucosa {v} mg/dL - {_nombre_pac(uid)}")
        guardar_sesion(uid)
        return (
            f"✅ Glucosa registrada [{fecha}]\n"
            f"{v:.0f} mg/dL | Estado: {estado_cl} | Total registros: {total}"
        )
    except ValueError as e:
        return f"❌ {e}"
    except Exception as e:
        return f"❌ Error inesperado al registrar glucosa: {e}"

### 3 Registrar medicamento

In [ ]:
@tool
def registrar_medicamento(datos_med: str) -> str:
    """
    Registra que el paciente tomó un medicamento (losartan, metformina, etc.)
    Úsala cuando el usuario diga que el paciente se tomó las pastillas,
    que tomó un medicamento concreto, o mencione nombres como losartan,
    metformina, insulina, enalapril, amlodipino, glibenclamida u otro fármaco.
    El parámetro debe ser el nombre del medicamento, seguido opcionalmente de
    la dosis y frecuencia, separados por comas.
    Ejemplos de input válido:
      'losartan'
      'metformina, 500mg, cada 12 horas'
      'ibuprofeno, 400mg, cada 8 horas'
    Si el usuario solo menciona el nombre sin dosis ni frecuencia, usa solo el nombre.
    """
    try:
        uid   = _USER_ID_CTX.get()
        datos = _datos_paciente(uid)
        partes = [p.strip() for p in datos_med.split(",")]
        nombre = partes[0].lower() if partes else ""
        dosis  = partes[1] if len(partes) > 1 else "-"
        freq   = partes[2] if len(partes) > 2 else "-"

        if not nombre:
            return "❌ Indica el nombre del medicamento."

        fecha = ahora()
        datos["medicamentos"].append({
            "fecha"     : fecha,
            "nombre"    : nombre,
            "dosis"     : dosis,
            "frecuencia": freq,
        })
        total = len(datos["medicamentos"])
        logger.info(f"Medicamento '{nombre}' {dosis} - {freq} - {_nombre_pac(uid)}")
        guardar_sesion(uid)
        return (
            f"✅ {nombre.capitalize()} {dosis} registrado [{fecha}]\n"
            f"Frecuencia: {freq} | Total tomas registradas: {total}"
        )
    except ValueError as e:
        return f"❌ {e}"
    except Exception as e:
        return f"❌ Error inesperado al registrar medicamento: {e}"

### 4 Registrar síntoma

In [ ]:
@tool
def registrar_sintoma(datos_sint: str) -> str:
    """
    Registra un síntoma del paciente activo.
    Úsala cuando el usuario mencione mareos, fatiga, visión borrosa,
    dolor de cabeza, náuseas, palpitaciones u otras molestias.
    El parámetro es el síntoma seguido de su intensidad separados por coma.
    Intensidades válidas: leve, moderado, severo.
    Ejemplos:
      'mareos, leve'
      'dolor de cabeza, moderado'
      'palpitaciones, severo'
    CRÍTICO: Si el usuario NO indicó la intensidad, NO llames esta herramienta.
    Primero pregunta: ¿Es leve, moderado o severo?
    Solo si la llamas sin intensidad confirmada, usa 'no_especificada'.
    """
    try:
        uid   = _USER_ID_CTX.get()
        datos = _datos_paciente(uid)
        partes     = [p.strip() for p in datos_sint.split(",")]
        sintoma    = partes[0].capitalize() if partes else ""
        intensidad = partes[1].lower() if len(partes) > 1 else "no_especificada"

        if not sintoma:
            return "❌ Indica el síntoma."

        mapa = {
            "leve"            : "🟡 Leve",
            "moderado"        : "🟠 Moderado",
            "severo"          : "🔴 Severo",
            "no_especificada" : "⚪ No especificada",
        }
        intensidad_fmt = mapa.get(intensidad, "⚪ No especificada")

        fecha = ahora()
        datos["sintomas"].append({
            "fecha"     : fecha,
            "sintoma"   : sintoma,
            "intensidad": intensidad_fmt,
        })

        alerta_txt = ""
        if intensidad_fmt == "🔴 Severo":
            _registrar_alerta("sintoma_severo", f"Síntoma severo: {sintoma} en {fecha}", True, uid)
            alerta_txt = "\n🚨 Síntoma severo registrado como alerta crítica."

        aviso = ""
        if intensidad_fmt == "⚪ No especificada":
            aviso = "\n⚠️ Intensidad no confirmada. Considera preguntar al usuario (leve/moderado/severo)."

        logger.info(f"Síntoma '{sintoma}' ({intensidad_fmt}) - {_nombre_pac(uid)}")
        guardar_sesion(uid)
        return f"✅ {sintoma} ({intensidad_fmt}) registrado [{fecha}]{alerta_txt}{aviso}"
    except ValueError as e:
        return f"❌ {e}"
    except Exception as e:
        return f"❌ Error inesperado al registrar síntoma: {e}"

### 5 Consultar historial

In [ ]:
@tool
def consultar_historial(tipo: str) -> str:
    """
    Consulta el historial de signos vitales, medicamentos o síntomas del paciente activo.
    Úsala cuando el usuario pregunte cómo ha estado el paciente, cuál fue el promedio
    de presión o glucosa, qué medicamentos tomó, o qué síntomas ha tenido.
    El parámetro indica qué tipo de historial consultar:
      'presion'      → últimas mediciones de presión arterial con promedios
      'glucosa'      → últimas mediciones de glucosa con promedios
      'medicamentos' → lista de medicamentos tomados
      'sintomas'     → síntomas registrados
      'resumen'      → resumen general de todos los datos (usa este para '¿cómo ha estado?')
    """
    try:
        uid    = _USER_ID_CTX.get()
        datos  = _datos_paciente(uid)
        nombre = _nombre_pac(uid)
        t      = tipo.strip().lower()

        if t == "presion":
            rp = datos["presion"]
            if not rp:
                return f"Sin registros de presión para {nombre}."
            sis = [r["sistolica"]  for r in rp]
            dia = [r["diastolica"] for r in rp]
            lineas = [f"📊 Presión arterial - {nombre} ({len(rp)} registros)"]
            lineas.append(f"Promedio: {sum(sis)/len(sis):.0f}/{sum(dia)/len(dia):.0f} mmHg")
            lineas.append(f"Rango: {min(sis)}-{max(sis)}/{min(dia)}-{max(dia)} mmHg")
            for r in rp[-5:]:
                lineas.append(f"  [{r['fecha']}] {r['sistolica']}/{r['diastolica']} mmHg")
            return "\n".join(lineas)

        elif t == "glucosa":
            rg = datos["glucosa"]
            if not rg:
                return f"Sin registros de glucosa para {nombre}."
            vals = [r["valor"] for r in rg]
            lineas = [f"📊 Glucosa - {nombre} ({len(rg)} registros)"]
            lineas.append(f"Promedio: {sum(vals)/len(vals):.0f} mg/dL")
            lineas.append(f"Rango: {min(vals):.0f}–{max(vals):.0f} mg/dL")
            for r in rg[-5:]:
                lineas.append(f"  [{r['fecha']}] {r['valor']:.0f} mg/dL")
            return "\n".join(lineas)

        elif t == "medicamentos":
            rm = datos["medicamentos"]
            if not rm:
                return f"Sin registros de medicamentos para {nombre}."
            lineas = [f"💊 Medicamentos - {nombre} ({len(rm)} tomas)"]
            for r in rm[-10:]:
                lineas.append(f"  [{r['fecha']}] {r['nombre'].capitalize()} {r['dosis']} c/{r['frecuencia']}")
            return "\n".join(lineas)

        elif t == "sintomas":
            rs = datos["sintomas"]
            if not rs:
                return f"Sin síntomas registrados para {nombre}."
            lineas = [f"🤒 Síntomas - {nombre} ({len(rs)} registros)"]
            for s in rs[-10:]:
                lineas.append(f"  [{s['fecha']}] {s['sintoma']} - {s['intensidad']}")
            return "\n".join(lineas)

        else:  # resumen
            rp = datos["presion"]; rg = datos["glucosa"]
            rm = datos["medicamentos"]; rs = datos["sintomas"]
            ra = [a for a in datos["alertas"] if a["critica"]]
            lineas = [f"📋 Resumen semanal - {nombre}"]
            lineas.append(f"Presión: {len(rp)} mediciones" + (
                f" | Promedio: {sum(r['sistolica'] for r in rp)/len(rp):.0f}/{sum(r['diastolica'] for r in rp)/len(rp):.0f} mmHg"
                if rp else " - sin datos"
            ))
            lineas.append(f"Glucosa: {len(rg)} mediciones" + (
                f" | Promedio: {sum(r['valor'] for r in rg)/len(rg):.0f} mg/dL"
                if rg else " - sin datos"
            ))
            lineas.append(f"Medicamentos registrados: {len(rm)} tomas")
            lineas.append(f"Síntomas registrados: {len(rs)}")
            lineas.append(f"Alertas críticas: {len(ra)}")
            return "\n".join(lineas)

    except ValueError as e:
        return f"❌ {e}"
    except Exception as e:
        return f"❌ Error inesperado al consultar historial: {e}"

### 6 Evaluar riesgo clínico

In [ ]:
# HERRAMIENTA 6 - Evaluar riesgo clínico
@tool
def evaluar_riesgo_clinico(vacio: str = "") -> str:
    """
    Evalúa el nivel de riesgo clínico del paciente activo basándose en sus registros.
    Úsala cuando el usuario pregunte por el estado de salud general, si hay alertas,
    si los valores son peligrosos, o si hay riesgo de complicaciones.
    El parámetro puede dejarse vacío o con cualquier texto; no se usa.
    Ejemplos que deben activar esta herramienta:
      '¿está bien el abuelo?'
      '¿hay riesgo?'
      'evaluación de riesgo'
      '¿cómo están sus valores?'
    """
    try:
        uid    = _USER_ID_CTX.get()
        datos  = _datos_paciente(uid)
        nombre = _nombre_pac(uid)
        rp, rg = datos["presion"], datos["glucosa"]

        if not rp and not rg:
            return "Sin datos suficientes para evaluar. Registra mediciones primero."

        alertas, nivel_riesgo = [], "Bajo"

        if rp:
            ep  = [r for r in rp if r["sistolica"] >= UMBRAL_SISTOLICA_MAX or r["diastolica"] >= UMBRAL_DIASTOLICA_MAX]
            pct = (len(ep) / len(rp)) * 100
            if len(ep) >= ALARMAS_NECESARIAS:
                nivel_riesgo = "Alto"
                fechas = ", ".join(r["fecha"] for r in ep[-3:])
                alertas.append(
                    f"🔴 HIPERTENSIÓN REITERADA\n"
                    f"  {len(ep)}/{len(rp)} mediciones superaron {UMBRAL_SISTOLICA_MAX}/{UMBRAL_DIASTOLICA_MAX} mmHg ({pct:.0f}%)\n"
                    f"  Últimos: {fechas}\n  Consulta médica urgente."
                )
            elif ep:
                nivel_riesgo = "Moderado"
                alertas.append(f"🟠 Presión elevada ocasional - {len(ep)} episodio(s). Vigilar.")

        if rg:
            hiper = [r for r in rg if r["valor"] >= UMBRAL_GLUCOSA_MAX]
            hipo  = [r for r in rg if r["valor"] < UMBRAL_GLUCOSA_MIN]
            if len(hiper) >= ALARMAS_NECESARIAS:
                nivel_riesgo = "Alto"
                pct = (len(hiper) / len(rg)) * 100
                alertas.append(
                    f"🔴 HIPERGLUCEMIA REITERADA\n"
                    f"  {len(hiper)}/{len(rg)} mediciones ≥ {UMBRAL_GLUCOSA_MAX} mg/dL ({pct:.0f}%)\n"
                    f"  Revisar dieta y dosis."
                )
            elif hiper:
                if nivel_riesgo != "Alto": nivel_riesgo = "Moderado"
                alertas.append(f"🟠 Glucosa elevada ocasional - {len(hiper)} episodio(s).")
            if len(hipo) >= 2:
                nivel_riesgo = "Alto"
                alertas.append(f"🔵 HIPOGLUCEMIA REITERADA - {len(hipo)} episodio(s) < 70 mg/dL.")

        sint_sev = [s for s in datos.get("sintomas", []) if s["intensidad"] == "🔴 Severo"]
        if sint_sev:
            if nivel_riesgo != "Alto": nivel_riesgo = "Moderado"
            alertas.append(f"🔴 Síntomas severos registrados - {len(sint_sev)} episodio(s).")

        iconos = {"Bajo": "🟢", "Moderado": "🟠", "Alto": "🔴"}
        msg = (
            f"Evaluación de riesgo - {nombre}\n"
            f"━━━━━━━━━━━━━━━━━━━\n"
            f"Nivel global: {iconos[nivel_riesgo]} {nivel_riesgo}\n\n"
        )
        msg += ("Alertas:\n\n" + "\n\n".join(alertas)) if alertas else "Sin patrones de riesgo detectados."
        logger.info(f"Riesgo [{nombre}]: {nivel_riesgo}")
        return msg

    except ValueError as e:
        return f"❌ {e}"
    except Exception as e:
        return f"❌ Error inesperado al evaluar riesgo: {e}"

### 7 Generar informe médico con gráfico

In [ ]:
# HERRAMIENTA 7 - Generar informe médico con gráfico
@tool
def generar_informe_medico(vacio: str = "") -> str:
    """
    Genera el informe clínico completo del paciente activo y guarda el gráfico.
    Úsala cuando el usuario pida el informe para el médico, el reporte clínico,
    el resumen para el doctor, o solicite generar/exportar el informe.
    El parámetro puede dejarse vacío; no se usa.
    IMPORTANTE: devuelve el texto completo del informe como resultado.
    Tras recibirlo como Observation, pásalo DIRECTAMENTE al Final Answer sin modificar.
    NO vuelvas a llamar esta herramienta en el mismo turno.
    """
    try:
        uid    = _USER_ID_CTX.get()
        datos  = _datos_paciente(uid)
        nombre = _nombre_pac(uid)
        rp     = datos["presion"]
        rg     = datos["glucosa"]
        rm     = datos["medicamentos"]
        rs     = datos.get("sintomas", [])
        ra     = datos.get("alertas", [])
        perfil = datos.get("perfil", {})

        if not rp and not rg:
            return "Sin mediciones para generar informe. Registra datos primero."

        inf  = f"INFORME CLÍNICO - {nombre.upper()}\n"
        inf += f"Generado: {ahora()}\n{'═'*36}\n\n"
        inf += "PERFIL DEL PACIENTE\n"
        inf += f"  Edad: {perfil.get('edad','-')} | Diagnósticos: {perfil.get('diagnosticos','-')}\n"
        inf += f"  Medicación permanente: {perfil.get('medicacion_perm','-')}\n\n"

        if rp:
            vs, vd = [r["sistolica"] for r in rp], [r["diastolica"] for r in rp]
            ps, pd = sum(vs)/len(vs), sum(vd)/len(vd)
            hta    = [r for r in rp if r["sistolica"] >= UMBRAL_SISTOLICA_MAX or r["diastolica"] >= UMBRAL_DIASTOLICA_MAX]
            inf += f"PRESIÓN ARTERIAL ({len(rp)} mediciones)\n"
            inf += f"Promedio: {ps:.0f}/{pd:.0f} mmHg | {len(hta)} episodio(s) hipertensivo(s)\n"
            for r in rp:
                m = " ⬆HTA" if r["sistolica"] >= UMBRAL_SISTOLICA_MAX or r["diastolica"] >= UMBRAL_DIASTOLICA_MAX else ""
                inf += f"  [{r['fecha']}] {r['sistolica']}/{r['diastolica']} mmHg{m}\n"
            inf += "\n"

        if rg:
            vg = [r["valor"] for r in rg]
            pg = sum(vg)/len(vg)
            inf += f"GLUCOSA EN SANGRE ({len(rg)} mediciones)\n"
            inf += f"Promedio: {pg:.0f} mg/dL | Rango: {min(vg):.0f}–{max(vg):.0f} mg/dL\n"
            for r in rg:
                m = " ⬆HIPER" if r["valor"] >= UMBRAL_GLUCOSA_MAX else (" ⬇HIPO" if r["valor"] < UMBRAL_GLUCOSA_MIN else "")
                inf += f"  [{r['fecha']}] {r['valor']:.0f} mg/dL{m}\n"
            inf += "\n"
        else:
            inf += "GLUCOSA EN SANGRE (0 mediciones)\n"
            inf += "  Sin registros de glucosa.\n\n"

        inf += f"MEDICAMENTOS ({len(rm)} toma(s))\n"
        for r in rm:
            inf += f"  [{r['fecha']}] {r['nombre'].capitalize()} {r['dosis']} c/{r['frecuencia']}\n"
        if not rm: inf += "  Sin registros.\n"
        inf += "\n"

        inf += f"SÍNTOMAS ({len(rs)} registro(s))\n"
        for s in rs:
            inf += f"  [{s['fecha']}] {s['sintoma']} - {s['intensidad']}\n"
        if not rs: inf += "  Sin síntomas registrados.\n"
        inf += "\n"

        criticas = [a for a in ra if a["critica"]]
        inf += f"ALERTAS ({len(ra)} total, {len(criticas)} crítica(s))\n"
        for a in ra[-5:]:
            tag = " 🚨CRÍTICA" if a["critica"] else ""
            inf += f"  [{a['fecha']}] {a['tipo']}: {a['mensaje']}{tag}\n"
        if not ra: inf += "  Sin alertas.\n"

        inf += f"\n{'─'*36}\n"
        inf += f"Ref OMS/ADA: Presión {UMBRAL_SISTOLICA_MIN}-{UMBRAL_SISTOLICA_MAX}/{UMBRAL_DIASTOLICA_MIN}-{UMBRAL_DIASTOLICA_MAX} mmHg | "
        inf += f"Glucosa {UMBRAL_GLUCOSA_MIN}-{UMBRAL_GLUCOSA_MAX} mg/dL\n"

        # Generar y guardar el gráfico en disco.
        # manejar_mensaje lo detecta por ruta_grafico(chat_id) y lo envía por Telegram.
        chat_id = _get_chat_id_actual()
        _generar_y_guardar_grafico(datos, nombre, chat_id)
        logger.info(f"Informe generado - {nombre} - chat_id={chat_id}")
        # Devolver el texto completo del informe como output de la herramienta.
        # LangChain lo pasa directamente al Final Answer, y manejar_mensaje
        # lo envía con reply_text. Simple y sin intermediarios.
        return inf

    except ValueError as e:
        return f"❌ {e}"
    except Exception as e:
        return f"❌ Error inesperado al generar informe: {e}"


# ContextVar y ruta de gráfico
def ruta_grafico(chat_id: int) -> str:
    """Retorna la ruta del PNG temporal exclusivo de un chat."""
    return f"/content/grafico_{chat_id}.png"


def _generar_y_guardar_grafico(datos: dict, nombre: str, chat_id: int = -1) -> str:
    # FIX: chat_id recibido como parámetro explícito para evitar colisión entre usuarios
    """Genera el gráfico clínico y lo guarda en archivo por chat_id."""
    rg, rp = datos.get("glucosa", []), datos.get("presion", [])
    if not rg and not rp:
        return "Sin datos para graficar."

    n_plots = (1 if rg else 0) + (2 if rp else 0)
    fig, axes = plt.subplots(n_plots, 1, figsize=(11, 5 * n_plots))
    fig.patch.set_facecolor("#F8F9FA")
    if n_plots == 1:
        axes = [axes]
    idx = 0

    if rg:
        ax   = axes[idx]; idx += 1
        vals = [r["valor"] for r in rg]
        xs   = list(range(len(vals)))
        lsup = max(vals + [UMBRAL_GLUCOSA_MAX + 50])
        ax.axhspan(UMBRAL_GLUCOSA_MAX, lsup,               alpha=0.12, color="red",   label=f"Alto (≥{UMBRAL_GLUCOSA_MAX})")
        ax.axhspan(UMBRAL_GLUCOSA_MIN, UMBRAL_GLUCOSA_MAX,  alpha=0.08, color="green", label="Normal")
        ax.axhspan(0, UMBRAL_GLUCOSA_MIN,                   alpha=0.12, color="blue",  label=f"Bajo (<{UMBRAL_GLUCOSA_MIN})")
        ax.plot(xs, vals, linewidth=2, marker="o", markersize=7, zorder=5)
        ax.set_xticks(xs)
        ax.set_xticklabels([r["fecha"] for r in rg], rotation=30, ha="right", fontsize=7)
        ax.set_ylabel("mg/dL"); ax.set_ylim(0, lsup)
        ax.set_title("Glucosa", fontsize=12, fontweight="bold")
        ax.legend(fontsize=8, loc="upper left"); ax.grid(True, linestyle=":", alpha=0.5)

    if rp:
        fec = [r["fecha"] for r in rp]
        sis = [r["sistolica"]  for r in rp]
        dia = [r["diastolica"] for r in rp]
        xs  = list(range(len(sis)))

        ax_s = axes[idx]; idx += 1
        lsup_s = max(sis + [UMBRAL_SISTOLICA_MAX + 20])
        ax_s.axhspan(UMBRAL_SISTOLICA_MAX, lsup_s,                alpha=0.15, color="red",   label=f"Alto (≥{UMBRAL_SISTOLICA_MAX})")
        ax_s.axhspan(UMBRAL_SISTOLICA_MIN, UMBRAL_SISTOLICA_MAX,   alpha=0.10, color="green", label="Normal")
        ax_s.axhspan(0, UMBRAL_SISTOLICA_MIN,                      alpha=0.15, color="blue",  label=f"Bajo (<{UMBRAL_SISTOLICA_MIN})")
        ax_s.plot(xs, sis, marker="o", linewidth=2, markersize=7, zorder=5)
        ax_s.set_xticks(xs); ax_s.set_xticklabels(fec, rotation=30, ha="right", fontsize=7)
        ax_s.set_ylabel("mmHg"); ax_s.set_ylim(0, lsup_s)
        ax_s.set_title("Presión sistólica", fontsize=12, fontweight="bold")
        ax_s.legend(fontsize=8, loc="upper left"); ax_s.grid(True, linestyle=":", alpha=0.5)

        ax_d = axes[idx]; idx += 1
        lsup_d = max(dia + [UMBRAL_DIASTOLICA_MAX + 20])
        ax_d.axhspan(UMBRAL_DIASTOLICA_MAX, lsup_d,                 alpha=0.15, color="red",   label=f"Alto (≥{UMBRAL_DIASTOLICA_MAX})")
        ax_d.axhspan(UMBRAL_DIASTOLICA_MIN, UMBRAL_DIASTOLICA_MAX,  alpha=0.10, color="green", label="Normal")
        ax_d.axhspan(0, UMBRAL_DIASTOLICA_MIN,                      alpha=0.15, color="blue",  label=f"Bajo (<{UMBRAL_DIASTOLICA_MIN})")
        ax_d.plot(xs, dia, linewidth=2, marker="o", markersize=7, zorder=5)
        ax_d.set_xticks(xs); ax_d.set_xticklabels(fec, rotation=30, ha="right", fontsize=7)
        ax_d.set_ylabel("mmHg"); ax_d.set_ylim(0, lsup_d)
        ax_d.set_title("Presión diastólica", fontsize=12, fontweight="bold")
        ax_d.legend(fontsize=8, loc="upper left"); ax_d.grid(True, linestyle=":", alpha=0.5)

    fig.suptitle(f"Paciente: {nombre}", fontsize=11, fontweight="bold", color="#333333", y=0.998)
    plt.tight_layout(pad=3.0)
    effective_chat_id = chat_id if chat_id != -1 else _get_chat_id_actual()
    ruta = ruta_grafico(effective_chat_id)
    plt.savefig(ruta, format="png", dpi=130, bbox_inches="tight", facecolor="#F8F9FA")
    plt.close(fig)
    logger.info(f"Gráfico guardado en {ruta}")
    return ruta

### 8 Gestión de pacientes

Herramienta que permite al agente gestionar la lista de pacientes del usuario.
El agente la invoca cuando detecta intenciones de gestión en el mensaje natural:

| Intención del usuario | Input a la herramienta |
|----------------------|------------------------|
| "¿Qué pacientes tengo?" | `listar` |
| "Quiero agregar a Juan" | `agregar: Juan` |
| "Cambia al paciente María" | `cambiar: María` |
| "Elimina a Carlos" | `eliminar: Carlos` (previa confirmación del usuario) |
| "¿Quién es el paciente activo?" | `activo` |


In [ ]:
# HERRAMIENTA 8 - Gestión de pacientes
@tool
def gestionar_pacientes(accion: str) -> str:
    """
    Gestiona la lista de pacientes del usuario: listar, crear, cambiar y eliminar.
    Úsala cuando el usuario quiera:
      - Ver qué pacientes tiene registrados: 'listar'
      - Agregar o crear un paciente nuevo: 'agregar: Nombre'
      - Cambiar al paciente activo: 'cambiar: Nombre'
      - Eliminar un paciente: 'eliminar: Nombre'
      - Ver cuál es el paciente actualmente activo: 'activo'
    Ejemplos de input válido:
      'listar'
      'agregar: Juan Pérez'
      'cambiar: María García'
      'eliminar: Carlos López'
      'activo'
    IMPORTANTE: para eliminar un paciente, el usuario debe confirmar explícitamente
    antes de que llames esta herramienta con 'eliminar'.
    """
    try:
        uid    = _USER_ID_CTX.get()
        accion = " ".join(accion.split()).lower()

        # ── listar ────────────────────────────────────────────────────────────
        if accion == "listar" or accion == "lista":
            pacientes = listar_pacientes(uid)
            activo    = _sesion(uid).get("paciente_activo")
            if not pacientes:
                return "No hay pacientes registrados. Puedes crear uno con 'agregar: Nombre'."
            lineas = ["👥 Pacientes registrados:"]
            for p in pacientes:
                marca = " ← activo" if p == activo else ""
                lineas.append(f"  • {p}{marca}")
            return "\n".join(lineas)

        # ── activo ────────────────────────────────────────────────────────────
        elif accion == "activo":
            activo = _sesion(uid).get("paciente_activo")
            if not activo:
                return "No hay paciente activo. Usa 'cambiar: Nombre' para seleccionar uno."
            return f"👤 Paciente activo: {activo}"

        # ── agregar ───────────────────────────────────────────────────────────
        elif accion.startswith("agregar:") or accion.startswith("crear:") or accion.startswith("nuevo:"):
            nombre = accion.split(":", 1)[1].strip()
            if not nombre:
                return "❌ Debes indicar el nombre del paciente. Ejemplo: 'agregar: Juan Pérez'"
            es_nuevo, nombre_norm = crear_paciente(nombre, uid)
            _sesion(uid)["paciente_activo"] = nombre_norm
            guardar_sesion(uid)
            if es_nuevo:
                return (
                    f"✅ Paciente '{nombre_norm}' creado y seleccionado como activo.\n"
                    f"Ya puedes registrar sus datos clínicos."
                )
            else:
                return f"ℹ️ El paciente '{nombre_norm}' ya existe. Ahora está seleccionado como activo."

        # ── cambiar ───────────────────────────────────────────────────────────
        elif accion.startswith("cambiar:") or accion.startswith("seleccionar:") or accion.startswith("activar:"):
            nombre = accion.split(":", 1)[1].strip()
            if not nombre:
                return "❌ Debes indicar el nombre. Ejemplo: 'cambiar: María García'"
            exito, nombre_norm = seleccionar_paciente(nombre, uid)
            if exito:
                guardar_sesion(uid)
                datos = _datos_paciente(uid)
                return (
                    f"✅ Paciente activo cambiado a: {nombre_norm}\n"
                    f"Registros: {len(datos['presion'])}P | {len(datos['glucosa'])}G | "
                    f"{len(datos['medicamentos'])}M | {len(datos.get('sintomas',[]))}S"
                )
            else:
                pacientes = listar_pacientes(uid)
                lista = ", ".join(pacientes) if pacientes else "(ninguno)"
                return (
                    f"❌ No se encontró el paciente '{nombre_norm}'.\n"
                    f"Pacientes disponibles: {lista}"
                )

        # ── eliminar ──────────────────────────────────────────────────────────
        elif accion.startswith("eliminar:") or accion.startswith("borrar:"):
            nombre = accion.split(":", 1)[1].strip()
            if not nombre:
                return "❌ Debes indicar el nombre. Ejemplo: 'eliminar: Carlos López'"
            nombre_norm = nombre.strip().title()
            # FIX: verificar flag de confirmación previa antes de ejecutar eliminación
            ses_actual = _sesion(uid)
            pendiente = ses_actual.get("_pendiente_eliminar")
            if pendiente != nombre_norm:
                # Primera solicitud: registrar intención y pedir confirmación
                ses_actual["_pendiente_eliminar"] = nombre_norm
                return (
                    f"⚠️ ¿Confirmas que deseas eliminar al paciente '{nombre_norm}'?\n"
                    f"Todos sus datos clínicos serán borrados permanentemente.\n"
                    f"Responde 'sí, eliminar a {nombre_norm}' para confirmar, o cualquier otra cosa para cancelar."
                )
            # Segunda solicitud confirmada: ejecutar eliminación
            ses_actual.pop("_pendiente_eliminar", None)
            exito = eliminar_paciente(nombre_norm, uid)
            if exito:
                guardar_sesion(uid)
                nuevo_activo = _sesion(uid).get("paciente_activo")
                msg = f"🗑️ Paciente '{nombre_norm}' eliminado correctamente."
                if nuevo_activo:
                    msg += f"\nPaciente activo ahora: {nuevo_activo}"
                else:
                    msg += "\nNo quedan pacientes. Crea uno con 'agregar: Nombre'."
                return msg
            else:
                pacientes = listar_pacientes(uid)
                lista = ", ".join(pacientes) if pacientes else "(ninguno)"
                return (
                    f"❌ No se encontró el paciente '{nombre_norm}' para eliminar.\n"
                    f"Pacientes disponibles: {lista}"
                )

        elif any(accion.startswith(p) for p in ("listar","lista","activo","agregar:","crear:","nuevo:","cambiar:","seleccionar:","activar:")):
            uid_clear = _USER_ID_CTX.get()
            if uid_clear != -1:
                _sesion(uid_clear).pop("_pendiente_eliminar", None)

        else:
            return (
                "❌ Acción no reconocida. Opciones válidas:\n"
                "  • 'listar' → ver todos los pacientes\n"
                "  • 'activo' → ver el paciente seleccionado\n"
                "  • 'agregar: Nombre' → crear nuevo paciente\n"
                "  • 'cambiar: Nombre' → cambiar paciente activo\n"
                "  • 'eliminar: Nombre' → eliminar paciente"
            )

    except Exception as e:
        return f"❌ Error inesperado en gestión de pacientes: {e}"

### 9 Registrar perfil del paciente

In [ ]:
# HERRAMIENTA 9 - Registrar perfil del paciente
@tool
def registrar_perfil(datos_perfil: str) -> str:
    """
    Registra el perfil clínico fijo del paciente: edad, diagnósticos,
    medicación permanente.
    Úsala cuando el usuario mencione datos estables como la edad del
    paciente, sus enfermedades crónicas o los medicamentos que toma siempre.
    Ejemplos:
      'tiene 78 años'
      '72 años, diabetes e hipertensión'
      'toma losartan y metformina de forma permanente'
    """
    try:
        uid    = _USER_ID_CTX.get()
        datos  = _datos_paciente(uid)
        nombre = _nombre_pac(uid)
        if "perfil" not in datos:
            datos["perfil"] = {}
        perfil = datos["perfil"]
        texto  = datos_perfil.strip()
        cambios = []

        m_edad = _re.search(r'\b(\d{2,3})\s*a[ñn]os?\b', texto, _re.IGNORECASE)
        if m_edad:
            perfil["edad"] = m_edad.group(1) + " años"
            cambios.append(f"Edad: {perfil['edad']}")

        m_diag = _re.search(
            r'(?:diagnósticos?|enfermedades?|padece|sufre\s+de|tiene)[:\s]+([^,\.]+(?:[,y][^,\.]+)*)',
            texto, _re.IGNORECASE
        )
        if m_diag:
            perfil["diagnosticos"] = m_diag.group(1).strip().capitalize()
            cambios.append(f"Diagnósticos: {perfil['diagnosticos']}")
        elif not m_edad:
            clinicos = ["diabetes","hipertensión","hipertension","cardíaca","cardiaca",
                        "renal","pulmonar","alzheimer","parkinson","artritis","anemia","asma"]
            if any(p in texto.lower() for p in clinicos):
                perfil["diagnosticos"] = texto.capitalize()
                cambios.append(f"Diagnósticos: {perfil['diagnosticos']}")

        m_med = _re.search(
            r'(?:toma|medicación\s+permanente|medicamentos?\s+(?:fijos?|permanentes?))[:\s]+([^\.]+)',
            texto, _re.IGNORECASE
        )
        if m_med:
            perfil["medicacion_perm"] = m_med.group(1).strip().capitalize()
            cambios.append(f"Medicación permanente: {perfil['medicacion_perm']}")

        if not cambios:
            perfil["diagnosticos"] = texto.capitalize()
            cambios.append(f"Info guardada: {perfil['diagnosticos']}")

        guardar_sesion(uid)
        logger.info(f"Perfil actualizado - {nombre}: {cambios}")
        return (
            f"✅ Perfil de {nombre} actualizado:\n  " + "\n  ".join(cambios) + "\n\n"
            f"Perfil completo:\n"
            f"  Edad: {perfil.get('edad','-')}\n"
            f"  Diagnósticos: {perfil.get('diagnosticos','-')}\n"
            f"  Medicación permanente: {perfil.get('medicacion_perm','-')}"
        )
    except ValueError as e:
        return f"❌ {e}"
    except Exception as e:
        return f"❌ Error al registrar perfil: {e}"

### Resumen

In [ ]:
HERRAMIENTAS = [
    registrar_presion,
    registrar_glucosa,
    registrar_medicamento,
    registrar_sintoma,
    consultar_historial,
    evaluar_riesgo_clinico,
    generar_informe_medico,
    gestionar_pacientes,
    registrar_perfil,
]

## LLM, Prompt y AgentExecutor

In [ ]:
# LLM utilizado
if not os.environ.get("OPENROUTER_API_KEY"):
    raise EnvironmentError(
        "❌ OPENROUTER_API_KEY vacía.\n"
    )

LLM = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    model="openai/gpt-4o-mini",
    temperature=0,
    max_tokens=1200
)

# Prompt ReAct - actualizado para incluir gestión de pacientes
PROMPT_TEMPLATE = """
Eres MediBot, un AGENTE CLÍNICO DE REGISTRO Y CONSULTA
para cuidadores, familiares o personal de salud.

Asistes en el REGISTRO, CONSULTA y GENERACIÓN DE INFORMES
de pacientes con hipertensión y/o diabetes.

⚠️ CONTEXTO CRÍTICO (SALUD):
- NO inventes datos.
- NO asumas información médica.
- NO repitas registros ya realizados.
- Todo informe o dato clínico DEBE venir de una herramienta.

════════════════════════════════════
REGLAS GENERALES (OBLIGATORIAS)
════════════════════════════════════

1. INTERPRETACIÓN DEL LENGUAJE
- El usuario escribe en lenguaje natural, incompleto o redundante.
- Infiere la intención usando el contexto previo.
- Nunca respondas “¿de qué me hablas?”.
- Si algo es ambiguo, pide aclaración breve.

2. UNA INTENCIÓN → UNA ACCIÓN
- Cada mensaje genera:
  a) UNA acción de herramienta
  b) O una respuesta directa
- Nunca ejecutes la misma acción dos veces con el mismo dato.
- Si el dato ya existe, CONFÍRMALO, no lo registres de nuevo.

3. NO REPETICIÓN (CRÍTICO)
- Si una presión, síntoma o medicamento ya fue registrado
  con el mismo valor:
  → NO lo registres otra vez.
  → Responde confirmando que ya está registrado.

4. MEDICINA RESPONSABLE
- No asumas intensidades.
- Si el usuario no indica gravedad:
  → Pregunta brevemente antes de registrar.
- “No tomó medicamentos” NO es un registro de medicación.

5. CONTROL DE FLUJO
- No generes loops.
- No repitas explicaciones.
- No pidas datos ya proporcionados.
- Nunca respondas con estados internos (“ya fue generado”).

════════════════════════════════════
REGLA CRÍTICA - PERFIL DEL PACIENTE
════════════════════════════════════

El PERFIL DEL PACIENTE contiene SOLO:
- Edad
- Diagnósticos crónicos
- Medicación permanente

CUÁNDO usar registrar_perfil:
- Si el usuario menciona edad:
  “tiene 78 años”, “72 años”
- Si menciona enfermedades crónicas:
  “diabetes”, “hipertensión”, “padece…”
- Si menciona medicación permanente:
  “toma losartán todos los días”

REGLAS OBLIGATORIAS:
- SI el usuario menciona cualquiera de estos datos → DEBES usar registrar_perfil.
- NO registres estos datos como síntomas.
- NO los registres como eventos.
- registrar_perfil NO genera informes ni gráficas.
- Si el perfil no ha sido registrado, el informe mostrará “-” correctamente.
- NO inventes ni completes datos que el usuario no haya dicho.

════════════════════════════════════
REGLA CRÍTICA - GESTIÓN DE PACIENTES
════════════════════════════════════

Las acciones de gestión de pacientes son DETERMINÍSTICAS.

- Para:
  • agregar paciente
  • cambiar paciente
  • listar pacientes
  • ver paciente activo

DEBES:
- Ejecutar la herramienta gestionar_pacientes DIRECTAMENTE.
- NO pedir confirmación.
- NO hacer preguntas previas.
- NO explicar lo obvio.

PROHIBIDO:
- Preguntar “¿confirmas que deseas agregar…?”
- Pedir validación adicional para crear o cambiar pacientes.

EXCEPCIÓN:
- SOLO la acción “eliminar paciente” requiere confirmación,
  y esa confirmación la maneja EXCLUSIVAMENTE la herramienta,
  NO el modelo.

FORMATO ESTRICTO DE Action Input

Cuando uses gestionar_pacientes:
- El Action Input DEBE ser UNA SOLA LÍNEA.
- Sin texto adicional.
- Sin explicaciones.
- Sin comillas extra.
Ejemplo válido:
  Action Input: agregar: Joao

════════════════════════════════════
REGLA CRÍTICA - NO REINTENTAR GESTIÓN DE PACIENTES
════════════════════════════════════

Si una llamada a gestionar_pacientes devuelve un mensaje de error (❌):
- NO vuelvas a llamar la herramienta en el mismo turno.
- NO intentes “arreglar” el input.
- NO pidas confirmación.
- RESPONDE al usuario MOSTRANDO el mensaje de la herramienta.

Está PROHIBIDO reintentar gestionar_pacientes más de una vez por mensaje.
════════════════════════════════════
USO DE HERRAMIENTAS
════════════════════════════════════

- Usa una herramienta SOLO si es necesario.
- Máximo UNA herramienta por mensaje.
- Si no hay acción clara, responde en texto.

════════════════════════════════════
REGLA CRÍTICA - INFORME MÉDICO
════════════════════════════════════

CUANDO el usuario pida:
- “informe clínico”
- “ver informe”
- “muéstrame el informe”
- “dame el informe”

DEBES:
1️⃣ Llamar a generar_informe_medico UNA SOLA VEZ.
2️⃣ Esperar el Observation con el TEXTO COMPLETO del informe.
3️⃣ RESPONDER MOSTRANDO ÍNTEGRAMENTE ese texto en el Final Answer.

PROHIBIDO ABSOLUTO:
- Decir “el informe ya fue generado”.
- Resumir el informe.
- Ocultarlo.
- Volver a llamar generar_informe_medico si ya se ejecutó correctamente.
- Inventar contenido del informe.

SI el usuario vuelve a pedir el informe:
→ Vuelve a GENERARLO y MUÉSTRALO COMPLETO.

════════════════════════════════════
FORMATO DE RESPUESTA (OBLIGATORIO)
════════════════════════════════════

Usa EXACTAMENTE uno de estos bloques por turno.

BLOQUE A - cuando usas herramienta:
Thought: razonamiento breve
Action: nombre exacto de la herramienta
Action Input: parámetro

BLOQUE B - cuando respondes al usuario:
Thought: Ya tengo la información necesaria para responder.
Final Answer: respuesta clara, humana y COMPLETA en español.

- Nunca muestres Thought/Action al usuario final.
- Nunca entregues respuestas duplicadas.

════════════════════════════════════
REGLAS ANTI-LOOP (CRÍTICAS)
════════════════════════════════════

- Si una herramienta responde con éxito (✅):
  → pasa DIRECTAMENTE a Final Answer.
- Nunca repitas la misma Action con el mismo Input.
- Una sola herramienta por mensaje.

════════════════════════════════════
HERRAMIENTAS DISPONIBLES
════════════════════════════════════
{tools}

Herramientas válidas:
{tool_names}

════════════════════════════════════
CONTEXTO
════════════════════════════════════

Historial:
{chat_history}

Pregunta del usuario:
{input}

{agent_scratchpad}"""

PROMPT = PromptTemplate.from_template(PROMPT_TEMPLATE)

def _manejar_error_parser(error) -> str:
    """
    FIX v9: maneja los tres casos de salida malformada del LLM.

    Caso A - LangChain detectó Action + Final Answer simultáneos:
      Extrae el Final Answer y lo devuelve directo.

    Caso B - Final Answer presente pero sin formato Thought/Action:
      Extrae y devuelve el Final Answer.

    Caso C - El LLM escribió una pregunta o texto plano sin formato ReAct
      (el bug principal: ocurre cuando el LLM quiere pedir confirmación
      o clarificación pero olvida usar Final Answer:).
      → Si el texto parece una pregunta válida (termina en ? o contiene
        palabras clave de confirmación), lo devolvemos directamente como
        respuesta al usuario. Esto evita el loop de "Could not parse".
      → Si el texto parece un resultado clínico (✅, ❌, mmHg, mg/dL),
        lo devolvemos directamente.
      → En cualquier otro caso, devolvemos un mensaje de error genérico.
    """
    texto = str(error)

    # ── Caso A: Action + Final Answer simultáneos ─────────────────────────────
    if "produced both a final answer and a parse-able action" in texto:
        if "Final Answer:" in texto:
            parte = texto.split("Final Answer:")[-1].strip()
            parte = parte.replace("`", "").strip()
            for corte in ["\nThought:", "\nAction:", "\n\n"]:
                if corte in parte:
                    parte = parte.split(corte)[0].strip()
            if parte:
                return parte
        return "Ya procesaste esta acción. Proporciona únicamente Final Answer: con la respuesta para el usuario, sin ningún Action."

    # ── Caso B: Final Answer presente pero mal formateado ────────────────────
    if "Final Answer:" in texto:
        parte = texto.split("Final Answer:")[-1].strip()
        parte = parte.replace("`", "").strip()
        if parte:
            return parte

    # ── Limpiar prefijo de LangChain ─────────────────────────────────────────
    texto_limpio = (texto
        .replace("Could not parse LLM output: `", "")
        .replace("`", "")
        .strip()
    )

    # ── Caso C1: texto clínico directo (resultado de herramienta) ────────────
    if any(s in texto_limpio for s in ["✅", "❌", "mmHg", "mg/dL"]):
        for marcador in ["✅", "❌", "He registrado", "Presión registrada"]:
            if marcador in texto_limpio:
                idx = texto_limpio.index(marcador)
                return texto_limpio[idx:].strip()
        return texto_limpio

    # ── Caso C2 (FIX PRINCIPAL): el LLM escribió una pregunta/confirmación ───
    # sin usar Final Answer:. Lo detectamos y lo devolvemos directamente
    # como respuesta al usuario en vez de entrar en loop.
    es_pregunta = (
        texto_limpio.endswith("?")
        or "¿" in texto_limpio
        or any(kw in texto_limpio.lower() for kw in [
            "confirmas", "confirma", "intensidad", "leve", "moderado", "severo",
            "eliminar a", "¿deseas", "¿quieres", "especifica", "indic"
        ])
    )
    if es_pregunta and "Action:" not in texto_limpio and len(texto_limpio) > 5:
        logger.info(f"[PARSER] Pregunta del LLM sin formato ReAct - devolviendo directo: {repr(texto_limpio[:100])}")
        return texto_limpio

    # ── Caso C3: texto plano sin Action (respuesta narrativa) ─────────────────
    if "Action:" not in texto_limpio and len(texto_limpio) > 10:
        return texto_limpio

    # ── Fallback ──────────────────────────────────────────────────────────────
    if len(texto_limpio) < 5:
        logger.warning(f"[PARSER] Respuesta vacía/irreconocible del LLM: {repr(texto[:200])}")
        return "Lo siento, no recibí respuesta del modelo. Por favor intenta de nuevo."

    logger.warning(f"[PARSER] Respuesta no parseada: {repr(texto_limpio[:200])}")
    return "Lo siento, no pude procesar esa respuesta. Por favor intenta de nuevo."

def crear_agente(memoria, chat_id: int = -1) -> "AgentExecutor":
    """
    FIX: caché de AgentExecutor por chat_id.
    Crea el agente solo si no existe o si se fuerza recreación (ej. en /start).
    Evita race conditions por múltiples mensajes concurrentes del mismo usuario.
    """
    if chat_id != -1 and chat_id in AGENTES:
        return AGENTES[chat_id]
    agente = create_react_agent(LLM, HERRAMIENTAS, PROMPT)
    executor = AgentExecutor(
        agent=agente,
        tools=HERRAMIENTAS,
        memory=memoria,
        verbose=True,
        handle_parsing_errors=_manejar_error_parser,
        return_intermediate_steps=False,
        max_iterations=6,
        max_execution_time=30,
        early_stopping_method="force",
    )
    if chat_id != -1:
        AGENTES[chat_id] = executor
    return executor

## Handlers de Telegram

In [ ]:
def limpiar_markdown(texto: str) -> str:
    """
    Elimina caracteres Markdown que Telegram no puede parsear correctamente
    cuando vienen del LLM con formato incorrecto.
    Se usa send_message sin parse_mode para evitar BadRequest.
    """
    # Eliminar formato markdown estándar
    texto = _re.sub(r'\*\*(.+?)\*\*', r'\1', texto)
    texto = _re.sub(r'\*(.+?)\*',     r'\1', texto)
    texto = _re.sub(r'_(.+?)_',       r'\1', texto)
    texto = _re.sub(r'`(.+?)`',       r'\1', texto)
    texto = _re.sub(r'\\boxed\{(.+?)\}', r'\1', texto)
    resultado = texto.strip()
    # FIX: si la limpieza produjo string vacío (regex destructiva), devolver original
    if not resultado and texto.strip():
        logger.warning("[limpiar_markdown] La limpieza produjo string vacío, devolviendo original")
        return texto.strip()
    return resultado

async def cmd_start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """
    Comando /start:
      1. Carga los datos del usuario desde local (misma carpeta Usuarios/ que CasoC_bot).
      2. Reinicia la memoria conversacional LangChain del chat.
      3. Muestra los pacientes existentes o da la bienvenida al usuario nuevo.
    """
    chat_id = update.effective_chat.id
    user_id = update.effective_user.id

    # Cargar sesión de este usuario desde local
    tiene_datos = cargar_sesion(user_id)

    # Reiniciar memoria conversacional (nueva sesión de chat)
    if chat_id in MEMORIAS:
        MEMORIAS[chat_id].clear()
    # FIX: destruir agente cacheado al /start para evitar memoria corrupta
    AGENTES.pop(chat_id, None)

    # Guardar user_id en context.user_data para recuperarlo en manejar_mensaje
    context.user_data["user_id"] = user_id
    # Limpiar flag de espera de paciente
    context.user_data["esperando_paciente"] = False

    ses = _sesion(user_id)

    if tiene_datos and ses.get("pacientes"):
            nombres  = list(ses["pacientes"].keys())
            activo   = ses.get("paciente_activo")
            lista    = "\n".join(f"  • {n}{'  ← activo' if n == activo else ''}" for n in nombres)
            await update.message.reply_text(
                f"¡Hola de nuevo! Soy MediBot 🏥\n\n"
                f"Tus pacientes registrados:\n{lista}\n\n"
                f"Recuerda que puedes registrar datos directamente hablándome de forma natural (ej. 'tiene 120/80 de presión' o 'ya tomó metformina'), o pedirme:\n"
                f"  • 📋 Un resumen o historial\n"
                f"  • 📊 El informe médico para el doctor\n"
                f"  • ⚠️ Evaluar si hay algún riesgo\n"
                f"  • 👥 'cambiar a [Nombre]', 'agregar a [Nombre]' o 'eliminar a [Nombre]'\n\n"
                f"¿Qué anotamos hoy?"
            )
    else:
            await update.message.reply_text(
                "¡Hola! Soy MediBot 🏥, tu asistente virtual para el monitoreo de salud. Estoy aquí para ayudarte a llevar el control clínico de tus pacientes de forma sencilla; solo tienes que hablarme de manera natural.\n\n"
                "Aquí tienes todo lo que puedo hacer por ti:\n"
                "👥 Gestionar pacientes: 'agrega a María' o 'cambia a Juan'.\n"
                "🩺 Signos vitales: 'hoy tuvo 120/80 de presión' o 'su glucosa está en 105'.\n"
                "💊 Medicamentos: 'ya se tomó el losartán'.\n"
                "🤒 Síntomas: 'tiene mareos leves'.\n"
                "📋 Consultas: Pídeme un historial o resumen.\n"
                "⚠️ Evaluar riesgos: '¿cómo está el paciente?'\n"
                "📊 Informes: Pídeme el 'informe médico' y crearé un reporte con gráficas.\n\n"
                "No necesitas memorizar comandos, ¡solo háblame como lo harías con un asistente humano!\n\n"
                "Para comenzar, ¿con qué paciente vamos a trabajar hoy?\n"
                "Escribe el nombre del paciente para crearlo y comenzar:"
            )
            context.user_data["esperando_paciente"] = True

async def manejar_mensaje(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """
    Handler único para todos los mensajes de texto.
    Flujo:
      1. Si se espera nombre de paciente → crear/seleccionar y confirmar.
      2. Si no hay paciente activo → pedir /start.
      3. En cualquier otro caso → invocar el AgentExecutor.
    Los ContextVars _USER_ID_CTX y _CHAT_ID_CTX aíslan los datos por tarea asyncio,
    permitiendo múltiples usuarios simultáneos sin interferencias.
    """
    chat_id = update.effective_chat.id
    user_id = context.user_data.get("user_id") or update.effective_user.id
    texto   = update.message.text.strip()

    # Fijar ContextVars de esta tarea asyncio (aislamiento por usuario/chat)
    _CHAT_ID_CTX.set(chat_id)
    _USER_ID_CTX.set(user_id)
    # FIX v9: guardar chat_id en la sesión para que las herramientas @tool
    # puedan recuperarlo aunque se ejecuten en un thread pool donde
    # _CHAT_ID_CTX no se propaga correctamente.
    _sesion(user_id)["_chat_id"] = chat_id

    # ── Primer mensaje: selección/creación de paciente ────────────────────────
    if context.user_data.get("esperando_paciente"):
        es_nuevo, nombre = crear_paciente(texto, user_id)
        _sesion(user_id)["paciente_activo"] = nombre
        context.user_data["esperando_paciente"] = False

        guardar_sesion(user_id)   # Persistir inmediatamente en local

        if es_nuevo:
            msg = (
                f"✅ Paciente '{nombre}' creado y seleccionado.\n\n"
                f"Ahora puedes registrar sus datos con lenguaje natural.\n"
            )
        else:
            datos = _datos_paciente(user_id)
            msg = (
                f"✅ Paciente '{nombre}' seleccionado. Continuamos su seguimiento.\n"
                f"Registros: {len(datos['presion'])}P | {len(datos['glucosa'])}G | "
                f"{len(datos['medicamentos'])}M | {len(datos.get('sintomas',[]))}S"
            )
        await update.message.reply_text(msg)
        return

    # ── Verificar que hay paciente activo ─────────────────────────────────────
    ses = _sesion(user_id)
    if not ses.get("paciente_activo"):
        await update.message.reply_text(
            "⚠️ No hay paciente activo.\n"
            "Escribe /start para seleccionar o crear un paciente."
        )
        return

    # ── Invocar el AgentExecutor ──────────────────────────────────────────────
    nombre_pac_activo = ses["paciente_activo"]
    input_completo    = f"[Paciente activo: {nombre_pac_activo}]\n{texto}"

    memoria = get_memoria(chat_id)
    # FIX: usar agente cacheado (no crear uno nuevo en cada mensaje)
    agente  = crear_agente(memoria, chat_id)

    await update.message.reply_text("Procesando...")

    try:
        resultado = agente.invoke({"input": input_completo})
        # FIX v9: manejar output None, vacío o con texto interno de LangChain
        respuesta = resultado.get("output") if isinstance(resultado, dict) else None

        agente_fallo = (
            not respuesta
            or respuesta.strip() in ("", "Agent stopped due to iteration limit or time limit.")
        )

        if agente_fallo:
            respuesta = "⏱️ No pude completar la solicitud a tiempo. Intenta de nuevo o reformula tu mensaje."
            logger.warning(f"AgentExecutor sin output útil para user {user_id}: {resultado}")
            try:
                mem = get_memoria(chat_id)
                if hasattr(mem, "chat_memory") and mem.chat_memory.messages:
                    msgs = mem.chat_memory.messages
                    if len(msgs) >= 2:
                        mem.chat_memory.messages = msgs[:-2]
                        logger.info(f"[MEMORIA] Turno fallido eliminado del historial (user {user_id})")
            except Exception as e_mem:
                logger.warning(f"[MEMORIA] No se pudo limpiar turno fallido: {e_mem}")
        else:
            guardar_sesion(user_id)

    except TimeoutError:
        logger.error(f"Timeout en AgentExecutor para user {user_id}")
        respuesta = "⏱️ El procesamiento tardó demasiado. Por favor intenta de nuevo."
    except Exception as e:
        logger.error(f"Error en AgentExecutor (user {user_id}): {type(e).__name__}: {e}")
        respuesta = "Ocurrió un error al procesar tu solicitud. Intenta de nuevo."

    # ── Enviar respuesta de texto ─────────────────────────────────────────────
    respuesta_limpia = limpiar_markdown(respuesta)
    LIMITE = 4000
    if len(respuesta_limpia) <= LIMITE:
        await update.message.reply_text(respuesta_limpia)
    else:
        partes = [respuesta_limpia[i:i+LIMITE] for i in range(0, len(respuesta_limpia), LIMITE)]
        for parte in partes:
            await update.message.reply_text(parte)

    # ── Enviar gráfico si fue generado ────────────────────────────────────────
    ruta = ruta_grafico(chat_id)
    if os.path.exists(ruta):
        try:
            with open(ruta, "rb") as f:
                await update.message.reply_photo(
                    photo=InputFile(f, filename="grafico_clinico.png"),
                    caption=f"Gráfico clínico - {nombre_pac_activo}",
                )
        except Exception as e:
            logger.warning(f"No se pudo enviar el gráfico: {e}")
        finally:
            try:
                os.remove(ruta)
            except OSError:
                pass

print("Handlers de Telegram listos")

## Bucle conversacional local

In [ ]:
def bucle_local():
    """
    Bucle interactivo de prueba directo en Colab (sin Telegram).
    Escribe 'salir' para terminar.

    Comandos especiales:
      paciente: Nombre      → seleccionar/crear paciente
      listar pacientes      → ver todos los pacientes del usuario de prueba
      eliminar: Nombre      → eliminar paciente (pide confirmación)

    En modo local se usa user_id=0 como identificador ficticio para local.
    """
    USER_ID_LOCAL = -999
    CHAT_ID_LOCAL = -999
    _CHAT_ID_CTX.set(CHAT_ID_LOCAL)
    _USER_ID_CTX.set(USER_ID_LOCAL)

    print("=" * 55)
    print("  MEDIBOT - Modo prueba local (sin Telegram)")
    print("  Comandos especiales:")
    print("    paciente: <Nombre>  → seleccionar/crear paciente")
    print("    listar pacientes    → ver todos los pacientes")
    print("    eliminar: <Nombre>  → eliminar paciente")
    print("    salir               → terminar el bucle")
    print("=" * 55)

    # Intentar cargar datos previos desde local
    if cargar_sesion(USER_ID_LOCAL):
        nombres = listar_pacientes(USER_ID_LOCAL)
        activo  = _sesion(USER_ID_LOCAL).get("paciente_activo")
        print(f"✅ Datos cargados desde local: {', '.join(nombres)}")
        if activo:
            print(f"   Paciente activo: {activo}")
    else:
        print("ℹ️  Sin datos previos en local para usuario de prueba.")

    memoria_local = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=False,
    )

    while True:
        try:
            entrada = input("\n🧑 Cuidador: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSesión finalizada.")
            break

        if not entrada:
            continue
        if entrada.lower() == "salir":
            print("👋 Hasta luego.")
            break

        # ── Seleccionar/crear paciente ────────────────────────────────────────
        if entrada.lower().startswith("paciente:"):
            nombre = entrada.split(":", 1)[1].strip()
            _, nombre_norm = crear_paciente(nombre, USER_ID_LOCAL)
            _sesion(USER_ID_LOCAL)["paciente_activo"] = nombre_norm
            memoria_local.clear()
            guardar_sesion(USER_ID_LOCAL)
            print(f"✅ Paciente activo: {nombre_norm}")
            continue

        # ── Listar pacientes ──────────────────────────────────────────────────
        if "listar" in entrada.lower() and "paciente" in entrada.lower():
            pacientes = listar_pacientes(USER_ID_LOCAL)
            activo    = _sesion(USER_ID_LOCAL).get("paciente_activo")
            if pacientes:
                print("👥 Pacientes registrados:")
                for p in pacientes:
                    marca = " ← activo" if p == activo else ""
                    print(f"  • {p}{marca}")
            else:
                print("Sin pacientes registrados.")
            continue

        # ── Eliminar paciente ─────────────────────────────────────────────────
        if entrada.lower().startswith("eliminar:"):
            nombre = entrada.split(":", 1)[1].strip().title()
            confirm = input(f"⚠️  ¿Confirmas eliminar a '{nombre}'? (sí/no): ").strip().lower()
            if confirm in ("si", "sí", "s", "yes"):
                exito = eliminar_paciente(nombre, USER_ID_LOCAL)
                if exito:
                    guardar_sesion(USER_ID_LOCAL)
                    nuevo_activo = _sesion(USER_ID_LOCAL).get("paciente_activo")
                    print(f"🗑️  Paciente '{nombre}' eliminado.")
                    print(f"   Paciente activo ahora: {nuevo_activo or '(ninguno)'}")
                else:
                    print(f"❌ No se encontró el paciente '{nombre}'.")
            else:
                print("Cancelado. No se eliminó ningún paciente.")
            continue

        # ── Verificar paciente activo ─────────────────────────────────────────
        if not _sesion(USER_ID_LOCAL).get("paciente_activo"):
            print("⚠️  Selecciona primero un paciente con: paciente: <Nombre>")
            continue

        # ── Invocar el agente ─────────────────────────────────────────────────
        agente     = crear_agente(memoria_local, CHAT_ID_LOCAL)  # FIX: pasar chat_id
        nombre_pac = _sesion(USER_ID_LOCAL)["paciente_activo"]
        try:
            resultado = agente.invoke({
                "input": f"[Paciente activo: {nombre_pac}]\n{entrada}",
            })
            respuesta = resultado.get("output", "Sin respuesta.")
            guardar_sesion(USER_ID_LOCAL)
        except Exception as e:
            respuesta = f"Error: {e}"

        print(f"\n🤖 MediBot: {respuesta}")

        ruta = ruta_grafico(CHAT_ID_LOCAL)
        if os.path.exists(ruta):
            print(f"📊 [Gráfico guardado en: {ruta}]")
            display(IPImage(ruta))
            os.remove(ruta)

# Para ejecutar el bucle, descomenta la siguiente línea:
# bucle_local()

## Inicialización

Estructura de despliegue idéntica a la del bot original: `ApplicationBuilder`, `CommandHandler`, `MessageHandler`, `run_polling`.

In [ ]:
def Iniciar():
    if not TOKEN:
        print("❌ No hay token de Telegram. Agrega 'TELEGRAM_TOKEN' en los Secrets de Colab.")
        return

    app = (
        ApplicationBuilder()
        .token(TOKEN)
        .read_timeout(60)
        .write_timeout(60)
        .build()
    )

    app.add_handler(CommandHandler("start", cmd_start))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, manejar_mensaje))

    async def _run():
        """
        FIX E4: reemplaza run_polling() que causaba
        'An asyncio.Future, a coroutine or an awaitable is required'.
        Usa la API de bajo nivel de PTB para evitar conflicto con el
        event loop de Jupyter/nest_asyncio.
        """
        await app.initialize()
        await app.start()
        await app.updater.start_polling(
            drop_pending_updates=True,
        )
        print("✅ Agente inteligente en marcha ✅")
        try:
            await asyncio.sleep(float('inf'))
        except (KeyboardInterrupt, asyncio.CancelledError):
            pass
        finally:
            try:
                await app.updater.stop()
                await app.stop()
                await app.shutdown()
            except Exception:
                pass

    loop = asyncio.get_event_loop()
    try:
        loop.run_until_complete(_run())
    except KeyboardInterrupt:
        print("🛑 Agente detenido por el usuario.")
    except Exception as e:
        print(f"❌ Agente apagado: {e}")

In [ ]:
async def limpiar_sesion_telegram():
    bot = Bot(token=TOKEN)
    await bot.delete_webhook(drop_pending_updates=True)
    # Consumir updates pendientes para liberar el getUpdates anterior
    try:
        await bot.get_updates(offset=-1, timeout=1)
    except Exception:
        pass
    print("Sesión de Telegram limpiada")

asyncio.get_event_loop().run_until_complete(limpiar_sesion_telegram())

In [ ]:
Iniciar()